In [7]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [19]:
def iamc_committed(rep_out_df, filter_reg, scen_name, rename_reg, rename_model, var_list):
    rep_out_df = rep_out_df[rep_out_df["Region"] == filter_reg]
    rep_out_df["Scenario"] = scen_name
    rep_out_df["Region"] = rename_reg
    rep_out_df["Model"] = rename_model

    ind_foss = rep_out_df[rep_out_df["Variable"].isin(["Final Energy|Industry|Liquids|Coal","Final Energy|Industry|Liquids|Gas","Final Energy|Industry|Liquids|Oil"])]
    ind_foss["Variable"] = "Final Energy|Industry|Liquids|Fossil"
    ind_foss_sum = ind_foss.sum(numeric_only=True).to_dict()
    meta = ind_foss.iloc[0,3:].to_dict()
    ind_foss_sum.update(meta)
    ind_foss_df = pd.DataFrame([ind_foss_sum])
    rep_out_df = pd.concat([rep_out_df, ind_foss_df])

    rc_foss = rep_out_df[rep_out_df["Variable"].isin(["Final Energy|Residential and Commercial|Liquids|Coal","Final Energy|Residential and Commercial|Liquids|Gas","Final Energy|Residential and Commercial|Liquids|Oil"])]
    rc_foss["Variable"] = "Final Energy|Residential and Commercial|Liquids|Fossil"
    rc_foss_sum = rc_foss.sum(numeric_only=True).to_dict()
    meta = rc_foss.iloc[0,3:].to_dict()
    rc_foss_sum.update(meta)
    rc_foss_df = pd.DataFrame([rc_foss_sum])
    rep_out_df = pd.concat([rep_out_df, rc_foss_df])

    rc_foss = rep_out_df[rep_out_df["Variable"].isin(["Final Energy|Transportation|Liquids|Coal","Final Energy|Transportation|Liquids|Gas","Final Energy|Transportation|Liquids|Oil"])]
    rc_foss["Variable"] = "Final Energy|Transportation|Liquids|Fossil"
    rc_foss_sum = rc_foss.sum(numeric_only=True).to_dict()
    meta = rc_foss.iloc[0,3:].to_dict()
    rc_foss_sum.update(meta)
    rc_foss_df = pd.DataFrame([rc_foss_sum])
    rep_out_df = pd.concat([rep_out_df, rc_foss_df])

    rep_out_df = rep_out_df.drop_duplicates()

    rep_out_df = rep_out_df.replace("Final Energy|Industry|Solids|Coal", "Final Energy|Industry|Solids|Fossil")
    rep_out_df = rep_out_df.replace("Final Energy|Residential and Commercial|Solids|Coal", "Final Energy|Residential and Commercial|Solids|Fossil")
    rep_out_df = rep_out_df.replace("Final Energy|Transportation|Solids|Coal", "Final Energy|Transportation|Solids|Fossil")
    
    rep_out_df = rep_out_df[rep_out_df["Variable"].isin(var_list["variable"])]
    rep_out_df = rep_out_df[rep_out_df["Variable"] != "Capacity Additions|Electricity|Storage Capacity"]
    rep_out_df = rep_out_df[rep_out_df["Variable"] != "Trade"]

    rep_out_df.loc[rep_out_df["Unit"] == "US$2010/kW", "Unit"] = "USD_2010/kW"
    rep_out_df.loc[rep_out_df["Unit"] == "Mt NOx/yr", "Unit"] = "Mt NO2/yr"
    rep_out_df.loc[rep_out_df["Unit"] == "billion US$2010/yr", "Unit"] = "billion USD_2010/yr"
    rep_out_df.loc[rep_out_df["Unit"] == "US$2010/kW/yr", "Unit"] = "USD_2010/kW/yr"

    rep_out_df.to_excel(f"D:/COMMITTED/T3.2/modelResults/{scen_name}.xlsx", index=False)


In [ ]:
variables = pd.read_excel("D:/COMMITTED/Models/NEST-Pakistan/submission/committed-internal-template.xlsx", sheet_name="variable")
scenarios = ['CurPol', "NDC-uncond", 'NDC-cond', 'LTS']
for sn in scenarios:
    scen_rep = pd.read_excel(f"D:/COMMITTED/Models/NEST-Pakistan/output/MESSAGEix-Pakistan 1_{sn}.xlsx")
    # scen_rep = scen_rep.drop(columns=[2040, 2045, 2050, 2055, 2060, 2070])
    iamc_committed(scen_rep, "R12_PAK", sn, "Pakistan", "MESSAGEix-Pakistan 1", variables)